In [1]:
# ============================================================
# NewsGuard — Day 2: Text Preprocessing
# Cell 1: Mount Drive + Restore Day 1 Artifacts + Verification
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/NewsGuard"

TRAIN_PATH = os.path.join(BASE_DIR, "data/splits/train.csv")
VAL_PATH   = os.path.join(BASE_DIR, "data/splits/validation.csv")
TEST_PATH  = os.path.join(BASE_DIR, "data/splits/test.csv")

required_files = {
    "Train": TRAIN_PATH,
    "Validation": VAL_PATH,
    "Test": TEST_PATH
}

print("=" * 65)
print("NewsGuard — Day 2")
print("Drive Mount + Day 1 Artifact Verification")
print("=" * 65)

for name, path in required_files.items():
    print(f"{name:12}: {'FOUND' if os.path.exists(path) else 'MISSING'}")
    print(f"Path        : {path}")
    print("-" * 65)

if not all(os.path.exists(path) for path in required_files.values()):
    raise FileNotFoundError(
        "Required Day 1 split files are missing. "
        "Do not continue until the Day 1 artifacts are restored."
    )

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("\nLoaded datasets:")
print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")

print("\nColumns:")
print("Train:", train_df.columns.tolist())

print("\nLabel distribution:")
print("Train:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation:")
print(val_df["label"].value_counts().sort_index())

print("\nTest:")
print(test_df["label"].value_counts().sort_index())

# Basic integrity checks
assert len(train_df) == 31282
assert len(val_df) == 3910
assert len(test_df) == 3911

assert set(train_df["label"].unique()) == {0, 1}
assert set(val_df["label"].unique()) == {0, 1}
assert set(test_df["label"].unique()) == {0, 1}

assert train_df["content"].notna().all()
assert val_df["content"].notna().all()
assert test_df["content"].notna().all()

print("\n" + "=" * 65)
print("DAY 1 ARTIFACT VERIFICATION PASSED")
print("Ready to begin Day 2 preprocessing.")
print("=" * 65)

Mounted at /content/drive
NewsGuard — Day 2
Drive Mount + Day 1 Artifact Verification
Train       : FOUND
Path        : /content/drive/MyDrive/NewsGuard/data/splits/train.csv
-----------------------------------------------------------------
Validation  : FOUND
Path        : /content/drive/MyDrive/NewsGuard/data/splits/validation.csv
-----------------------------------------------------------------
Test        : FOUND
Path        : /content/drive/MyDrive/NewsGuard/data/splits/test.csv
-----------------------------------------------------------------

Loaded datasets:
Train      : (31282, 6)
Validation : (3910, 6)
Test       : (3911, 6)

Columns:
Train: ['title', 'text', 'label', 'content', 'word_count', 'char_count']

Label distribution:
Train:
label
0    14325
1    16957
Name: count, dtype: int64

Validation:
label
0    1791
1    2119
Name: count, dtype: int64

Test:
label
0    1791
1    2120
Name: count, dtype: int64

DAY 1 ARTIFACT VERIFICATION PASSED
Ready to begin Day 2 preprocessi

In [2]:
# ================================================================
# NewsGuard — Day 2
# Cell 2: Text Cleaning Function
# ================================================================

import re
import html
import unicodedata

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities
    text = html.unescape(text)

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", " ", text)

    # Convert to lowercase
    text = text.lower()

    # Replace control characters with spaces
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    # Normalize common dash characters
    text = re.sub(r"[‐-‒–—―]", "-", text)

    # Normalize quotation marks
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)

    # Keep letters, numbers, basic punctuation and spaces
    text = re.sub(r"[^a-z0-9\s.,!?;:'\"()\-]", " ", text)

    # Remove repeated punctuation
    text = re.sub(r"([!?.,;:])\1+", r"\1", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("Text cleaning function created successfully.")
print("Preprocessing is deterministic and reproducible.")

Text cleaning function created successfully.
Preprocessing is deterministic and reproducible.


In [3]:
# ================================================================
# NewsGuard — Day 2
# Cell 3: Apply Text Preprocessing
# ================================================================

for df in [train_df, val_df, test_df]:
    df["clean_content"] = df["content"].apply(clean_text)

print("Preprocessing applied successfully.")
print()
print("Dataset shapes:")
print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")
print()
print("New column:")
print("clean_content")

Preprocessing applied successfully.

Dataset shapes:
Train      : (31282, 7)
Validation : (3910, 7)
Test       : (3911, 7)

New column:
clean_content


In [5]:
# ================================================================
# NewsGuard — Day 2
# Cell 5: Inspect Empty Preprocessed Rows
# ================================================================

def inspect_empty_rows(df, name):
    empty_rows = df[df["clean_content"].str.strip() == ""]

    print(f"\n{name} empty rows: {len(empty_rows)}")
    print("=" * 64)

    for idx, row in empty_rows.iterrows():
        print(f"\nIndex : {idx}")
        print(f"Label : {row['label']}")
        print(f"Original content length : {len(str(row['content']))}")
        print("Original content:")
        print(repr(str(row["content"])[:1000]))
        print("-" * 64)

inspect_empty_rows(train_df, "TRAIN")
inspect_empty_rows(val_df, "VALIDATION")
inspect_empty_rows(test_df, "TEST")


TRAIN empty rows: 3

Index : 4890
Label : 0
Original content length : 149
Original content:
'https://fedup.wpengine.com/wp-content/uploads/2015/04/hillarystreetart.jpg https://fedup.wpengine.com/wp-content/uploads/2015/04/hillarystreetart.jpg'
----------------------------------------------------------------

Index : 12499
Label : 0
Original content length : 177
Original content:
'https://100percentfedup.com/video-hillary-asked-about-trump-i-just-want-to-eat-some-pie/ https://100percentfedup.com/video-hillary-asked-about-trump-i-just-want-to-eat-some-pie/'
----------------------------------------------------------------

Index : 19188
Label : 0
Original content length : 291
Original content:
'https://100percentfedup.com/served-roy-moore-vietnamletter-veteran-sets-record-straight-honorable-decent-respectable-patriotic-commander-soldier/ https://100percentfedup.com/served-roy-moore-vietnamletter-veteran-sets-record-straight-honorable-decent-respectable-patriotic-commander-soldier/'
-----

In [6]:
# ================================================================
# NewsGuard — Day 2
# Cell 6: Fix Text Cleaning Function
# ================================================================

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities
    text = html.unescape(text)

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Convert Markdown links to their visible text
    # [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Convert to lowercase
    text = text.lower()

    # Replace control characters with spaces
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    # Normalize dash characters
    text = re.sub(r"[‐-‒–—―]", "-", text)

    # Normalize quotation marks
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)

    # Keep letters, numbers, whitespace and useful punctuation
    text = re.sub(r"[^a-z0-9\s.,!?;:'\"()\-]", " ", text)

    # Remove repeated punctuation
    text = re.sub(r"([!?.,;:])\1+", r"\1", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Re-apply corrected preprocessing
for df in [train_df, val_df, test_df]:
    df["clean_content"] = df["content"].apply(clean_text)

print("Cleaning function updated successfully.")
print("Preprocessing re-applied to all datasets.")

Cleaning function updated successfully.
Preprocessing re-applied to all datasets.


In [8]:
# ================================================================
# NewsGuard — Day 2
# Cell 8: Final Text Cleaning Fix
# ================================================================

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    # Detect Markdown links
    markdown_links = re.findall(r"\[([^\]]+)\]\([^)]+\)", text)

    # If content contains only Markdown links whose visible text
    # is itself a URL, preserve it as a meaningful URL token
    temp = text.strip()

    if markdown_links:
        visible_text = " ".join(markdown_links).strip()

        if visible_text and re.fullmatch(
            r"(https?://\S+\s*)+",
            visible_text
        ):
            return "url"

        # Otherwise keep visible text
        text = re.sub(
            r"\[([^\]]+)\]\([^)]+\)",
            r"\1",
            text
        )

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace URLs with a token instead of deleting them completely
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " url ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " email ",
        text
    )

    # Lowercase
    text = text.lower()

    # Replace control characters
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    # Normalize dashes
    text = re.sub(r"[‐-‒–—―]", "-", text)

    # Normalize quotation marks
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)

    # Keep useful characters
    text = re.sub(
        r"[^a-z0-9\s.,!?;:'\"()\-]",
        " ",
        text
    )

    # Remove repeated punctuation
    text = re.sub(r"([!?.,;:])\1+", r"\1", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Re-apply preprocessing
for df in [train_df, val_df, test_df]:
    df["clean_content"] = df["content"].apply(clean_text)

print("Final cleaning function applied successfully.")
print("URL-only records are preserved using the 'url' token.")

Final cleaning function applied successfully.
URL-only records are preserved using the 'url' token.


In [9]:
# ================================================================
# NewsGuard — Day 2
# Cell 9: Final Preprocessing Validation
# ================================================================

for df, name in [
    (train_df, "TRAIN"),
    (val_df, "VALIDATION"),
    (test_df, "TEST")
]:
    empty_count = (df["clean_content"].str.strip() == "").sum()
    null_count = df["clean_content"].isna().sum()

    print(f"{name:<12} | Empty: {empty_count} | Null: {null_count}")

    assert empty_count == 0
    assert null_count == 0

print("\n" + "=" * 64)
print("PREPROCESSING VALIDATION PASSED")
print("=" * 64)

TRAIN        | Empty: 0 | Null: 0
VALIDATION   | Empty: 0 | Null: 0
TEST         | Empty: 0 | Null: 0

PREPROCESSING VALIDATION PASSED


In [11]:
# ================================================================
# NewsGuard — Day 2
# Cell 11: Inspect Remaining Raw URL
# ================================================================

for df, name in [
    (train_df, "TRAIN"),
    (val_df, "VALIDATION"),
    (test_df, "TEST")
]:
    mask = df["clean_content"].str.contains(
        r"https?://|www\.",
        regex=True,
        na=False
    )

    if mask.any():
        print(f"\n{name} — Remaining URL records: {mask.sum()}")
        print("=" * 64)

        for idx, row in df[mask].iterrows():
            print(f"Index : {idx}")
            print("Original:")
            print(repr(row["content"]))
            print("\nCleaned:")
            print(repr(row["clean_content"]))
            print("-" * 64)


TRAIN — Remaining URL records: 1
Index : 30222
Original:
'EXPOSE THE LIES: Shut Down Planned Parenthood’s Phone Lines TODAY IS  SCHEDULE YOUR MAMMOGRAM DAY  WITH PLANNED PARENTHOOD   PP LIES TO THE AMERICAN PEOPLE SO WE THINK IT S IMPORTANT TO CALL THEM OUT ON THE FACTS: They DO NOT perform mammograms!Please call your local Planned parenthood clinic and ask if you can schedule your mammogram with them   Do it today! A few years ago, Cecile Richards went on television and said that if Planned Parenthood was defunded, women would lose access to services  such as mammograms.  As a former clinic director of Planned Parenthood, I knew this was simply not true.We scheduled our first  Schedule Your Mammogram Day  after that. We had over 10,000 people call Planned Parenthood. While we were hoping for an official statement from Planned Parenthood, they were silent.Just a couple nights ago at the Miss America pageant, one of the contestants stated that Planned Parenthood shouldn t be defunded b

In [12]:
# ================================================================
# NewsGuard — Day 2
# Cell 12: Remove Remaining Raw URL Patterns
# ================================================================

def remove_remaining_urls(text):
    if not isinstance(text, str):
        return ""

    # Replace any remaining Markdown links with URL token
    text = re.sub(
        r"\[[^\]]*\]\(\s*https?://[^)]+\)",
        " url ",
        text,
        flags=re.IGNORECASE
    )

    # Replace any remaining raw URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " url ",
        text,
        flags=re.IGNORECASE
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


for df in [train_df, val_df, test_df]:
    df["clean_content"] = df["clean_content"].apply(remove_remaining_urls)

print("Remaining URL cleanup applied successfully.")

Remaining URL cleanup applied successfully.


In [13]:
# ================================================================
# NewsGuard — Day 2
# Cell 13: Final URL & Empty Text Validation
# ================================================================

all_clean_text = pd.concat([
    train_df["clean_content"],
    val_df["clean_content"],
    test_df["clean_content"]
], ignore_index=True)

for df, name in [
    (train_df, "TRAIN"),
    (val_df, "VALIDATION"),
    (test_df, "TEST")
]:
    empty_count = (df["clean_content"].str.strip() == "").sum()
    null_count = df["clean_content"].isna().sum()

    print(f"{name:<12} | Empty: {empty_count} | Null: {null_count}")

    assert empty_count == 0
    assert null_count == 0

remaining_urls = all_clean_text.str.contains(
    r"https?://|www\.",
    regex=True,
    na=False
).sum()

print("\n" + "-" * 64)
print(f"Remaining raw URLs : {remaining_urls}")
print("-" * 64)

assert remaining_urls == 0

print("\n" + "=" * 64)
print("FINAL TEXT PREPROCESSING VALIDATION PASSED")
print("=" * 64)

TRAIN        | Empty: 0 | Null: 0
VALIDATION   | Empty: 0 | Null: 0
TEST         | Empty: 0 | Null: 0

----------------------------------------------------------------
Remaining raw URLs : 0
----------------------------------------------------------------

FINAL TEXT PREPROCESSING VALIDATION PASSED


In [14]:
# ================================================================
# NewsGuard — Day 2
# Cell 14: Save Preprocessed Datasets
# ================================================================

import os

processed_dir = "/content/drive/MyDrive/NewsGuard/data/processed"
os.makedirs(processed_dir, exist_ok=True)

train_path = os.path.join(processed_dir, "train_preprocessed.csv")
val_path = os.path.join(processed_dir, "validation_preprocessed.csv")
test_path = os.path.join(processed_dir, "test_preprocessed.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("=" * 64)
print("PREPROCESSED DATASETS SAVED")
print("=" * 64)

print(f"\nTrain      : {train_path}")
print(f"Validation : {val_path}")
print(f"Test       : {test_path}")

print("\nShapes:")
print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")

PREPROCESSED DATASETS SAVED

Train      : /content/drive/MyDrive/NewsGuard/data/processed/train_preprocessed.csv
Validation : /content/drive/MyDrive/NewsGuard/data/processed/validation_preprocessed.csv
Test       : /content/drive/MyDrive/NewsGuard/data/processed/test_preprocessed.csv

Shapes:
Train      : (31282, 7)
Validation : (3910, 7)
Test       : (3911, 7)


In [15]:
# ================================================================
# NewsGuard — Day 2
# Cell 15: Final Day 2 Integrity Check
# ================================================================

required_files = [
    train_path,
    val_path,
    test_path
]

print("=" * 64)
print("NewsGuard — Day 2 Final Integrity Check")
print("=" * 64)

# File existence
for path in required_files:
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'} : {path}")
    assert os.path.exists(path)

# Required columns
required_columns = [
    "title",
    "text",
    "label",
    "content",
    "word_count",
    "char_count",
    "clean_content"
]

for df, name, expected_rows in [
    (train_df, "Train", 31282),
    (val_df, "Validation", 3910),
    (test_df, "Test", 3911)
]:
    assert len(df) == expected_rows
    assert list(df.columns) == required_columns
    assert df["label"].isin([0, 1]).all()
    assert df["content"].notna().all()
    assert df["clean_content"].notna().all()
    assert (df["clean_content"].str.strip() != "").all()
    assert df["clean_content"].str.contains(
        r"https?://|www\.",
        regex=True,
        na=False
    ).sum() == 0

    print(f"{name:<12}: PASSED")

# Final row count
assert len(train_df) + len(val_df) + len(test_df) == 39103

print("\n" + "=" * 64)
print("DAY 2 PREPROCESSING INTEGRITY CHECK PASSED")
print("Ready for Day 3 — TF-IDF Feature Engineering")
print("=" * 64)

NewsGuard — Day 2 Final Integrity Check
FOUND : /content/drive/MyDrive/NewsGuard/data/processed/train_preprocessed.csv
FOUND : /content/drive/MyDrive/NewsGuard/data/processed/validation_preprocessed.csv
FOUND : /content/drive/MyDrive/NewsGuard/data/processed/test_preprocessed.csv
Train       : PASSED
Validation  : PASSED
Test        : PASSED

DAY 2 PREPROCESSING INTEGRITY CHECK PASSED
Ready for Day 3 — TF-IDF Feature Engineering
